In [1]:
import numpy as np
import torch
from torch import nn, optim
from torch.nn import functional as F
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
from torchvision.transforms import Compose, Normalize, Resize, ToTensor
from train_cuda import train

In [2]:
# temp dataset
temp_transform = Compose([Resize(28), ToTensor()])
temp_dataset = ImageFolder(root="./Datasets/RPS/train", transform=temp_transform)

In [3]:
# No need to bother with shuffling, as this is not the data loader we’ll use to train the
# model anyway. We’ll use it to compute statistics only. By the way, we need
# statistics for each channel, as required by the Normalize() transform.
bs = 64
temp_loader = DataLoader(temp_dataset, batch_size=bs)

In [4]:
# statistics of each image per channel
results = []
for images, labels in temp_loader:
    # NCHW
    n_samples, n_channels, n_height, n_weight = images.size()
    # Flatten HW into a single dimension
    flatten_per_channel = images.reshape(n_samples, n_channels, -1)
    # Computes statistics of each image per channel
    # Average pixel value per channel
    # (n_samples, n_channels)
    means = flatten_per_channel.mean(dim=2)
    # Standard deviation of pixel values per channel
    # (n_samples, n_channels)
    stds = flatten_per_channel.std(dim=2)
    n_samples = torch.tensor([n_samples] * n_channels).float()
    # Adds up statistics of all images in a mini-batch
    # (1, n_channels)
    sum_means = means.sum(dim=0)
    sum_stds = stds.sum(dim=0)
    result = torch.stack([n_samples, sum_means, sum_stds])
    # Append results
    results.append(result)
results = torch.stack(results).sum(dim=0)


In [5]:
total_samples, total_means, total_stds = results
norm_mean = total_means / total_samples
norm_std = total_stds / total_samples

In [ ]:
size = 64
composer = Compose([Resize(size), ToTensor(), Normalize(norm_mean, norm_std)])
train_data = ImageFolder(root="./Datasets/RPS/train", transform=composer)
val_data = ImageFolder(root="./Datasets/RPS/test", transform=composer)
# Builds a loader of each set
bs = 64
train_loader = DataLoader(train_data, batch_size=bs, shuffle=True)
val_loader = DataLoader(val_data, batch_size=bs)

In [9]:
class RPScnn(nn.Module):
    def __init__(self, n_filters, p=0.0):
        super().__init__()
        self.n_filters = n_filters
        self.p = p
        # Creates the convolution layers
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=n_filters, kernel_size=3)
        self.conv2 = nn.Conv2d(
            in_channels=n_filters, out_channels=n_filters * 2, kernel_size=3
        )
        self.conv3 = nn.Conv2d(
            in_channels=n_filters * 2, out_channels=n_filters * 4, kernel_size=3
        )
        # Creates the linear layers
        self.fc1 = nn.Linear(n_filters * 4 * 6 * 6, 120)
        self.fc2 = nn.Linear(120, 50)
        self.fc3 = nn.Linear(50, 3)
        # Creates dropout layers
        self.drop = nn.Dropout(self.p)

    def featurizer(self, x):
        # First convolutional block
        # 3@64x64 -> n_filters@62x62 -> n_filters@31x31
        x = self.conv1(x)
        x = F.relu(x)
        x = F.max_pool2d(x, kernel_size=2)
        # Second convolutional block
        # n_filters@31x31 -> n_filters@29x29 -> n_filters@14x14
        x = self.conv2(x)
        x = F.relu(x)
        x = F.max_pool2d(x, kernel_size=2)
        # Third convolutional block
        # n_filters@14x14 -> n_filters@12x12 -> n_filters@6x6
        x = self.conv3(x)
        x = F.relu(x)
        x = F.max_pool2d(x, kernel_size=2)
        # Input dimension (n_filters*4@6x6)
        # Output dimension (n_filters * 4 * 6 * 6)
        x = nn.Flatten()(x)
        return x

    def classifier(self, x):
        # Classifier
        # Hidden Layer
        # Input dimension (n_feature * 4 * 6 * 6)
        # Output dimension (50)
        if self.p > 0:
            x = self.drop(x)
        x = self.fc1(x)
        x = F.relu(x)
        # Output Layer
        # Input dimension (50)
        # Output dimension (3)
        if self.p > 0:
            x = self.drop(x)
        x = self.fc2(x)
        x = F.relu(x)
        if self.p > 0:
            x = self.drop(x)
        x = self.fc3(x)
        return x

    def forward(self, x):
        x = self.featurizer(x)
        x = self.classifier(x)
        return x

In [10]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_rps = RPScnn(n_filters=6, p=0.3).to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer_rps = optim.Adam(model_rps.parameters(), lr=3e-4)
epochs = 50
lr = 3e-4
losses, val_losses = train(
    model_rps, train_loader, val_loader, loss_fn, optimizer_rps, epochs, device
)

Epoch 1/50 - loss: 1.0560 - val_loss: 1.0145
Epoch 2/50 - loss: 0.7195 - val_loss: 0.6772
Epoch 3/50 - loss: 0.3155 - val_loss: 0.4718
Epoch 4/50 - loss: 0.1607 - val_loss: 0.3507
Epoch 5/50 - loss: 0.0872 - val_loss: 0.2556
Epoch 6/50 - loss: 0.0703 - val_loss: 0.2436
Epoch 7/50 - loss: 0.0529 - val_loss: 0.1985
Epoch 8/50 - loss: 0.0502 - val_loss: 0.2039
Epoch 9/50 - loss: 0.0315 - val_loss: 0.2004
Epoch 10/50 - loss: 0.0233 - val_loss: 0.1615
Epoch 11/50 - loss: 0.0222 - val_loss: 0.1734
Epoch 12/50 - loss: 0.0178 - val_loss: 0.1547
Epoch 13/50 - loss: 0.0155 - val_loss: 0.1971
Epoch 14/50 - loss: 0.0124 - val_loss: 0.1941
Epoch 15/50 - loss: 0.0105 - val_loss: 0.1573
Epoch 16/50 - loss: 0.0146 - val_loss: 0.3104
Epoch 17/50 - loss: 0.0064 - val_loss: 0.2443
Epoch 18/50 - loss: 0.0065 - val_loss: 0.2713
Epoch 19/50 - loss: 0.0055 - val_loss: 0.3154
Epoch 20/50 - loss: 0.0036 - val_loss: 0.2838
Epoch 21/50 - loss: 0.0035 - val_loss: 0.2535
Epoch 22/50 - loss: 0.0066 - val_loss: 0.19

In [11]:
import torchmetrics

accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=3).to(device)
for x, y in val_loader:
    x, y = x.to(device), y.to(device)
    yhat = model_rps(x)
    accuracy(yhat, y)
accuracy.compute()
print(f"Test Accuracy: {accuracy.compute().item() * 100:.2f}%")

Test Accuracy: 92.20%
